### Imports and data

In [15]:
import pandas as pd
import numpy as np
from tqdm import tqdm
pd.set_option('display.max_colwidth', None)

captions = pd.read_csv("train_v1(3)/captions.csv")
captions.head(5)

,path,caption
0,caption_dataset/ea86708d8af74316b5010a910a758a47.jpg,a close up of a yellow motorcycle parked on a black surface
1,caption_dataset/0df33da31f1943caa5d1af0ba3e1d389.jpg,motorcycle parked on the grass near a curb with a dog sitting on the seat
2,caption_dataset/8acabd6cb1fa4299a3039bb4cd9d9480.jpg,a close up of a red motorcycle parked on a gray surface
3,caption_dataset/e3a52e3538444e47b50151f34ab07ed2.jpg,a close up of a motorcycle parked on a white surface
4,caption_dataset/e5b869522c4c4ffe873e370dddff05a5.jpg,arafed motorcycle parked in front of a garage door


In [16]:
airplane_labels = ['airplane']
car_labels = ['car']
cat_labels = ['cat', 'kitten']
dog_labels = ['dog']
fruit_labels = ['fruit', 'apple', 'lemon', 'pear', 'plum', 'orange', 'tomato']
flower_labels = ['flower']
motorbike_labels = ['motorbike', 'motorcycle', 'bike']
person_labels = ['person', 'man', 'woman', 'girl', 'boy', 'female', 'male', 'men', 'player']
all_labels = [airplane_labels, car_labels, cat_labels, dog_labels, flower_labels, fruit_labels, motorbike_labels, person_labels]

captions['label'] = captions['caption'].apply(
    lambda x: next((i for i, labels in enumerate(all_labels) if any(label in x for label in labels)), None)
)
captions['label'].value_counts()

label
7.0    352
1.0    325
2.0    311
4.0    263
6.0    261
5.0    245
3.0    220
0.0    175
Name: count, dtype: int64

In [17]:
captions = captions.dropna()
captions = captions.drop('caption', axis=1)

### Model

In [72]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from sklearn.model_selection import train_test_split

train_transforms = A.Compose([
    # A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=30, p=0.5),
    A.Resize(128, 128),
    # A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])
val_transforms = A.Compose([
    A.Resize(128, 128),
    # A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

class ImageDataset(Dataset):
    def __init__(self, paths, labels, transforms = None):
        self.paths = paths
        self.labels = labels
        self.transforms = transforms

    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, index):
        img = np.array(Image.open('train_v1(3)/' + self.paths[index]).convert('RGB'))
        if self.transforms:
            img = self.transforms(image = img)['image']
        img = img.float()
        return img, torch.tensor(self.labels[index], dtype=torch.long)
    
train_df, val_df = train_test_split(captions, test_size=0.1)
train_ds = ImageDataset(train_df['path'].tolist(), train_df['label'].tolist(), train_transforms)
val_ds = ImageDataset(val_df['path'].tolist(), val_df['label'].tolist(), val_transforms)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

In [76]:
from torchvision.models import efficientnet_b0

num_classes = 8
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = efficientnet_b0()
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
model = model.to(device)

In [ ]:
epochs = 50
early_stopping = 5

criterion = nn.CrossEntropyLoss()
optim = torch.optim.AdamW(model.parameters(), lr=5e-5)

no_improvment = 0
best_loss = float('inf')
best_model_state = None
for epoch in tqdm(range(1, epochs+1)):
    train_acc = 0
    train_loss = 0
    total = 0
    model.train()
    for (x, y) in train_loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optim.step()
        optim.zero_grad()

        preds = torch.argmax(logits, dim=1)
        train_acc += (preds==y).sum().item()
        train_loss += loss.item()
        total += y.size(0)

    train_acc /= total
    train_loss /= len(train_loader)

    val_acc, val_loss, total = 0, 0, 0
    model.eval()
    with torch.no_grad():
        for (x, y) in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            preds = torch.argmax(logits, dim=1)
            loss = criterion(logits, y)

            val_acc += (preds==y).sum().item()
            val_loss += loss.item()
            total += y.size(0)

        val_acc /= total
        val_loss /= len(val_loader)

    if(val_loss < best_loss):
        no_improvment = 0
        best_loss = val_loss
        best_model_state = model.state_dict()
    else:
        no_improvment+=1
        if no_improvment >= early_stopping:
            print('Early stopping')
            break

    print(f'Train loss: {train_loss} | Train acc: {train_acc} | Val loss: {val_loss} | Val acc: {val_acc}')

model.load_state_dict(best_model_state)

  2%|▏         | 1/50 [00:03<03:14,  3.98s/it]

Train loss: 1.8982251472160465 | Train acc: 0.2649793388429752 | Val loss: 2.116599832262312 | Val acc: 0.16666666666666666


  4%|▍         | 2/50 [00:07<03:06,  3.89s/it]

Train loss: 1.6921038373571928 | Train acc: 0.3672520661157025 | Val loss: 1.598657591002328 | Val acc: 0.36574074074074076


  6%|▌         | 3/50 [00:11<02:59,  3.81s/it]

...

 20%|██        | 10/50 [00:47<03:25,  5.13s/it]

Train loss: 0.5734985475657416 | Train acc: 0.8026859504132231 | Val loss: 0.9133495347840446 | Val acc: 0.6898148148148148


 20%|██        | 10/50 [00:53<03:32,  5.31s/it]

Early stopping


<All keys matched successfully>